# tiny-mlir on a T4

[tiny-mlir](https://github.com/sdiehl/tiny-mlir) is the small compiler from the [MLIR series](https://www.stephendiehl.com/posts/mlir_introduction/). Python functions become typed tensor graphs, MLIR fuses and bufferizes them, matmuls become a shared-memory tiled kernel and the remaining loops become CUDA kernels. The example runs pretrained GPT-2 on the GPU and checks every decoding step against the NumPy reference.

Select a GPU runtime first: Runtime, Change runtime type, T4 GPU. Any card of compute capability sm_75 or newer works.

In [ ]:
!nvidia-smi -L

## Install

The MLIR wheel is the CUDA build matching this Python version. NVIDIA's pip package supplies `ptxas` and `libdevice`, and `uv` runs the reference implementation in its own locked environment.

In [ ]:
import sys

tag = f"cp{sys.version_info.major}{sys.version_info.minor}"
wheel = (
    "https://github.com/makslevental/mlir-wheels/releases/download/latest/"
    f"mlir_python_bindings-22.0.0.2025112901%2Bcuda.9bae84b0-{tag}-{tag}-"
    "manylinux_2_27_x86_64.manylinux_2_28_x86_64.whl"
)
!git clone -q https://github.com/sdiehl/tiny-mlir
%cd -q tiny-mlir
!pip install -q "mlir_python_bindings@{wheel}" "nvidia-cuda-nvcc-cu12<13" "numpy>=2.1.2" safetensors tokenizers huggingface-hub uv
!python fetch_model.py

## A kernel

Any traced function can target the GPU. The compiled object keeps the module at each stage, so the kernels are visible after outlining and again as the PTX that `gpu-module-to-binary` embeds for `sm_75`.

In [ ]:
import numpy as np
from mlir import ir
from mlir.dialects.gpu import ObjectAttr
from mlir.passmanager import PassManager

from tinymlir import gpu, jit
from tinymlir.jit import BUFFERIZATION_PIPELINE, GPU
from tinymlir.ops import gelu

x = np.random.default_rng(0).normal(size=(64, 256)).astype(np.float32)
f = jit(gelu)
np.testing.assert_allclose(f(x, target=GPU), f(x), rtol=1e-5, atol=1e-5)

compiled = f.compile(x, target=GPU)
with compiled.context:
    PassManager.parse(BUFFERIZATION_PIPELINE).run(compiled.optimized.operation)
    gpu.outline(compiled.optimized)
print(compiled.optimized)

In [ ]:
binaries = []
with compiled.context:
    compiled.lowered.operation.walk(
        lambda op: (op.name == "gpu.binary" and binaries.append(op)) or ir.WalkResult.ADVANCE
    )
    ptx = ObjectAttr(binaries[0].attributes["objects"][0]).object.decode()
print(ptx[:1200])

## GPT-2

Weights stay resident in managed memory across calls. Each sequence length compiles its own specialization, so the first tokens pay for compilation and the rest run cached kernels.

In [ ]:
!python example.py "Alan Turing theorized that computers would one day become" --tokens 10 --gpu

## Checking the reference

`verify.py` fetches the pinned NumPy implementation, runs it with `uv` in its own locked environment, and compares the embeddings, both residual outputs of every block, the logits and the chosen token at each of ten steps.

In [ ]:
!python verify.py --gpu

## The whole model as one kernel

`--fused` traces the reference forward pass as written into a single kernel per padded sequence length, so fusion and the matmul substitution see every block at once. The first call compiles the whole model, which takes a few minutes.

In [ ]:
!python example.py "Alan Turing theorized that computers would one day become" --tokens 10 --gpu --fused